# Malayalam OCR: Dataset Generation and Two‑Headed CNN Training

Make this notebook your end‑to‑end workflow for building a small Malayalam OCR system:

- Generate a synthetic character dataset with multiple fonts and augmentations
- Train a two‑headed CNN that predicts consonant and vowel sign jointly
- Visualize class balance, track training curves, and evaluate with confusion matrices and sample predictions

## How to use this notebook

1. Generate dataset (images + labels.csv) — runs once or whenever you change fonts/augmentation
2. Load the labels and build TensorFlow datasets
3. Configure GPU/mixed precision for faster training
4. Build and train the two‑headed CNN with callbacks
5. Visualize training history and evaluate on validation set

Tip: If you already have `malayalam_dataset/images` and `malayalam_dataset/labels.csv`, you can skip the generation cell and jump to the training sections.

## Requirements (major)
- Python 3.9+
- Pillow, Albumentations, OpenCV, NumPy, Pandas, tqdm
- TensorFlow 2.x (with GPU support if available)
- scikit‑learn, Matplotlib, Seaborn

See `dataset/requirements.txt` for a complete list and versions.

## Project artifacts created by this notebook
- `malayalam_dataset/images/` — generated PNGs (original + augmented)
- `malayalam_dataset/labels.csv` — filename → (consonant, vowel)
- `malayalam_dataset/best_model_*.keras` — best checkpoints saved by callbacks
- `malayalam_dataset/training_log.csv` — per‑epoch metrics
- `malayalam_dataset/training_history.png` — loss/accuracy curves
- `malayalam_dataset/confusion_matrices.png` — consonant + vowel confusion matrices
- `malayalam_dataset/sample_predictions.png` — grid of validation samples with predictions

## Fonts
Ensure the following fonts exist under `dataset/fonts/` (already included in this repo). You can add more to increase diversity:
- AnekMalayalam, Chilanka, Gayathri, Manjari, NotoSansMalayalam, NotoSerifMalayalam

## Table of Contents
- Dataset Generation (Cell 2)
- Train a two‑headed CNN for Malayalam OCR (overview)
  - Load labels and create tf.data
  - GPU setup (mixed precision + memory growth)
  - Model architecture (two heads)
  - Training strategy (callbacks, LR schedule)
  - Training history and curves
  - Evaluation, confusion matrices, and sample predictions

Run cells from top to bottom. Safe to re‑run sections; outputs will be overwritten where applicable.

In [ ]:
import os
from PIL import Image, ImageDraw, ImageFont
import albumentations as A
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# ----------------------------
# CONFIG
# ----------------------------
IMG_SIZE = 128
FONT_SIZE = 65
OUTPUT_DIR = "malayalam_dataset"
IMG_DIR = os.path.join(OUTPUT_DIR, "images")
NUM_AUG_PER_IMAGE = 20  # how many augmented versions per clean image

# ----------------------------
# CHARACTERS (same as yours)
# ----------------------------
vowels = [
    "അ", "ആ", "ഇ", "ഈ", "ഉ", "ഊ", "ഋ", "എ", "ഏ", "ഐ", "ഒ", "ഓ", "ഔ"
]

consonants = [
    "ക", "ഖ", "ഗ", "ഘ", "ങ",
    "ച", "ഛ", "ജ", "ഝ", "ഞ",
    "ട", "ഠ", "ഡ", "ഢ", "ണ",
    "ത", "ഥ", "ദ", "ധ", "ന",
    "പ", "ഫ", "ബ", "ഭ", "മ",
    "യ", "ര", "ല", "വ", "ശ", "ഷ", "സ", "ഹ", "ള", "ഴ", "റ",
]

vowel_signs = [
    "ാ", "ി", "ീ", "ു", "ൂ", "ൃ", "െ", "േ", "ൈ", "ൊ", "ോ", "ൌ"
]

chillaksharams = ["ൻ", "ർ", "ൽ", "ൾ", "ൿ"]

consonant_vowels = [c + v for c in consonants for v in vowel_signs]
characters = vowels + consonants + chillaksharams + consonant_vowels

# Fonts (ensure present)
fonts = [
    "fonts/AnekMalayalam-VariableFont_wdth,wght.ttf",
    "fonts/Chilanka-Regular.ttf",
    "fonts/Gayathri-Regular.ttf",
    "fonts/Manjari-Regular.ttf",
    "fonts/NotoSansMalayalam-VariableFont_wdth,wght.ttf",
    "fonts/NotoSerifMalayalam-VariableFont_wght.ttf",
    "fonts/Ezhuthu-Regular.ttf",
    "fonts/RIT-Kutty-Bold.ttf"
]

# ----------------------------
# AUGMENTATIONS (grayscale-compatible, version-safe)
# ----------------------------
# Helper for stroke width variation using morphology.
# Accept **kwargs to be compatible with Albumentations Lambda (which may pass shape/rows/cols).
import random

def vary_stroke(img, p=0.7, **kwargs):
    if img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img.copy()
    
    if random.random() > p:
        return img

    # Choose operation with weighted probability
    op = random.choices(["erode", "dilate"], weights=[0.3, 0.7])[0]
    k = 1  # safe kernel radius
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*k+1, 2*k+1))
    
    if op == "erode":
        # Only erode if the text is thick enough
        if np.count_nonzero(gray) > 20:  
            gray = cv2.erode(gray, kernel, iterations=1)
    else:
        gray = cv2.dilate(gray, kernel, iterations=1)

    if img.ndim == 3:
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    return gray


# Use transforms that are broadly compatible with Albumentations 0.4+ and work on single-channel images
augment = A.Compose([
    # Geometric jitter similar to handwriting placement
    A.Rotate(limit=12, border_mode=cv2.BORDER_CONSTANT, border_value=255, p=0.9),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=0,
                       border_mode=cv2.BORDER_CONSTANT, border_value=255, p=0.9),

    # Perspective/warps (avoid deprecated args)
    A.Perspective(scale=(0.02, 0.06), keep_size=True, p=0.5),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.5),
    A.ElasticTransform(alpha=40, sigma=6, p=0.5),

    # Stroke width variation (custom morphology)
    # A.Lambda(image=vary_stroke, p=0.7),

    # Photometric variations common in scans/photos
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.2, p=0.6),
    A.RandomGamma(gamma_limit=(70, 130), p=0.4),
    A.MultiplicativeNoise(multiplier=(0.8, 1.2), elementwise=True, p=0.3),

    # Blur and JPEG artifacts (ImageCompression is widely supported)
    A.MotionBlur(blur_limit=5, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.ImageCompression(quality_lower=40, quality_upper=80, p=0.3),

    # # Random occlusions / ink gaps (use CoarseDropout for broader compatibility)
    # A.CoarseDropout(max_holes=3, max_height=IMG_SIZE // 8, max_width=IMG_SIZE // 8,
    #                 fill_value=255, p=0.25),
])

# ----------------------------
# HELPERS & RENDER
# ----------------------------
def render_character(char, font_path):
    font = ImageFont.truetype(font_path, size=FONT_SIZE)
    img = Image.new("L", (IMG_SIZE, IMG_SIZE), color=255)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), char, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    draw.text(((IMG_SIZE - w) / 2, (IMG_SIZE - h) / 2), char, fill=0, font=font)
    return np.array(img)

def decompose_char(char):
    if char in vowels:
        return ("VOWEL_" + char, "NONE")
    if char in chillaksharams:
        return (char, "NONE")
    if len(char) == 1 and char in consonants:
        return (char, "NONE")
    for c in consonants:
        for v in vowel_signs:
            if char == c + v:
                return (c, v)
    return (char, "NONE")

def codepoint_slug(s):
    """
    Turn a label like 'ക' or 'ാ' or 'VOWEL_അ' into a safe, unique ascii slug:
    - NONE -> 'NONE'
    - VOWEL_അ -> 'VOWEL_U0D05' (using hex codepoints)
    - ക     -> 'U0D15'
    - ാ     -> 'U0D3E'
    """
    if s == "NONE":
        return "NONE"
    if s.startswith("VOWEL_"):
        ch = s.split("_", 1)[1]
        return "VOWEL_" + "_".join(f"U{ord(c):04X}" for c in ch)
    # normal char(s) (could be single or multiple codepoints)
    return "_".join(f"U{ord(c):04X}" for c in s)

# ----------------------------
# MAIN (writing images + labels.csv)
# ----------------------------
os.makedirs(IMG_DIR, exist_ok=True)
labels = []  # (filename, consonant_label, vowel_label)

for char in tqdm(characters, desc="Generating dataset"):
    for font_path in fonts:
        font_name = os.path.splitext(os.path.basename(font_path))[0]
        try:
            img = render_character(char, font_path)
            consonant_label, vowel_label = decompose_char(char)

            # create a unique slug-based base_name that encodes both labels + font
            slug_c = codepoint_slug(consonant_label)
            slug_v = codepoint_slug(vowel_label)
            base_name = f"{slug_c}__{slug_v}__{font_name}"

            # original image
            fname_orig = f"{base_name}__orig.png"
            cv2.imwrite(os.path.join(IMG_DIR, fname_orig), img)
            labels.append((fname_orig, consonant_label, vowel_label))

            # augmented images
            for i in range(NUM_AUG_PER_IMAGE):
                aug_img = augment(image=img)["image"]
                fname_aug = f"{base_name}__aug{i}.png"
                cv2.imwrite(os.path.join(IMG_DIR, fname_aug), aug_img)
                labels.append((fname_aug, consonant_label, vowel_label))

        except Exception as e:
            print(f"Error rendering {char!r} with {font_path}: {e}")

# Save CSV
df = pd.DataFrame(labels, columns=["filename", "consonant", "vowel"])
df.to_csv(os.path.join(OUTPUT_DIR, "labels.csv"), index=False)

print("✅ Dataset generation complete!")


## New handwriting-style augmentations

To make the synthetic data closer to handwritten scans/photos, we expanded the augmentation pipeline:

- Affine with translate/rotate/shear: mimics slant and off-center placement
- Perspective + Elastic + GridDistortion: introduces non-linear warps similar to natural pen strokes and camera perspective
- Stroke width variation (morphology): randomly erodes/dilates to simulate thin/thick strokes and ink spread
- Photometric noise: brightness/contrast, gamma, ISO and multiplicative noise to reflect lighting/sensor variability
- Blur and compression: motion/gaussian blur, JPEG-like artifacts, and downscale/upsample to mimic low-res captures
- CoarseDropout: small occlusions or ink gaps

You can tune strength/probabilities or increase `NUM_AUG_PER_IMAGE` for more diversity. If training becomes unstable, slightly reduce the warp/noise probabilities.

## Train a Two‑Headed CNN for Malayalam OCR

This section trains a TensorFlow model with two classification heads:
- Head 1 (cons): predicts the base consonant class, or special tokens like chillaksharams and standalone vowels encoded as `VOWEL_*`
- Head 2 (vow): predicts the vowel sign (including `NONE`)

Inputs
- Images: `malayalam_dataset/images/*.png` generated above
- Labels: `malayalam_dataset/labels.csv` with columns `[filename, consonant, vowel]`

Outputs
- Best model checkpoints: `malayalam_dataset/best_model_*.keras`
- Logs and figures: training curves, confusion matrices, sample predictions

Process
1. Load labels, fit encoders, and analyze class balance
2. Create `tf.data` pipelines for training and validation
3. Configure GPU (optional): mixed precision + memory growth
4. Build a shared CNN backbone with two Dense heads
5. Train with callbacks (checkpointing, early stopping, LR schedule)
6. Visualize history and evaluate performance

If you have pre-generated data, you can start from here.

In [ ]:
# Imports for training
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from sklearn.metrics import confusion_matrix, classification_report
import json

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

IMG_DIR = os.path.join("malayalam_dataset", "images")
CSV_PATH = os.path.join("malayalam_dataset", "labels.csv")
ENCODINGS_PATH = os.path.join("malayalam_dataset", "encodings.json")
IMG_SIZE = 128
BATCH_SIZE = 64
SEED = 42

logger.info("Loading dataset...")
df = pd.read_csv(CSV_PATH)
print(df.head())
logger.info(f"Dataset loaded with {len(df)} samples")

# Fit label encoders
le_consonant = LabelEncoder()
le_vowel = LabelEncoder()

le_consonant.fit(df["consonant"])  # e.g., 'ക', 'NONE', 'VOWEL_U0D05', etc.
le_vowel.fit(df["vowel"])          # e.g., 'ാ', 'NONE', etc.

# Persist class encodings for reproducible inference
os.makedirs(os.path.dirname(ENCODINGS_PATH), exist_ok=True)
with open(ENCODINGS_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "consonant_classes": le_consonant.classes_.tolist(),
        "vowel_classes": le_vowel.classes_.tolist(),
        "note": "Exact class ordering used during training. Load this in inference to decode predictions."
    }, f, ensure_ascii=False, indent=2)
logger.info(f"Saved label encodings to {ENCODINGS_PATH}")

num_consonant_classes = len(le_consonant.classes_)
num_vowel_classes = len(le_vowel.classes_)

logger.info(f"Consonant classes: {num_consonant_classes}")
logger.info(f"Vowel classes: {num_vowel_classes}")
print("Consonant classes:", num_consonant_classes)
print("Vowel classes:", num_vowel_classes)

# Class balance analysis
def analyze_class_distribution(df):
    logger.info("Analyzing class distribution...")
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    
    # Consonant distribution
    cons_counts = df['consonant'].value_counts().sort_index()
    axes[0,0].bar(range(len(cons_counts)), cons_counts.values)
    axes[0,0].set_title('Consonant Class Distribution')
    axes[0,0].set_xlabel('Class Index')
    axes[0,0].set_ylabel('Count')
    axes[0,0].tick_params(axis='x', rotation=45, labelsize=8)
    
    # Vowel distribution
    vow_counts = df['vowel'].value_counts().sort_index()
    axes[0,1].bar(range(len(vow_counts)), vow_counts.values)
    axes[0,1].set_title('Vowel Class Distribution')
    axes[0,1].set_xlabel('Class Index')
    axes[0,1].set_ylabel('Count')
    axes[0,1].tick_params(axis='x', rotation=45, labelsize=8)
    
    # Top consonant classes
    top_cons = cons_counts.head(20)
    axes[1,0].barh(range(len(top_cons)), top_cons.values)
    axes[1,0].set_yticks(range(len(top_cons)))
    axes[1,0].set_yticklabels(top_cons.index, fontsize=10)
    axes[1,0].set_title('Top 20 Consonant Classes')
    axes[1,0].set_xlabel('Count')
    
    # Top vowel classes
    top_vow = vow_counts.head(15)
    axes[1,1].barh(range(len(top_vow)), top_vow.values)
    axes[1,1].set_yticks(range(len(top_vow)))
    axes[1,1].set_yticklabels(top_vow.index, fontsize=10)
    axes[1,1].set_title('Top 15 Vowel Classes')
    axes[1,1].set_xlabel('Count')
    
    plt.tight_layout()
    plt.savefig('malayalam_dataset/class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print(f"\nConsonant classes statistics:")
    print(f"Total classes: {len(cons_counts)}")
    print(f"Min count: {cons_counts.min()}")
    print(f"Max count: {cons_counts.max()}")
    print(f"Mean count: {cons_counts.mean():.1f}")
    print(f"Std count: {cons_counts.std():.1f}")
    
    print(f"\nVowel classes statistics:")
    print(f"Total classes: {len(vow_counts)}")
    print(f"Min count: {vow_counts.min()}")
    print(f"Max count: {vow_counts.max()}")
    print(f"Mean count: {vow_counts.mean():.1f}")
    print(f"Std count: {vow_counts.std():.1f}")
    
    return cons_counts, vow_counts

# Perform class balance analysis
cons_dist, vow_dist = analyze_class_distribution(df)

# Train/val split stratified on combined label for balance - INCREASED TO 15%
logger.info("Creating train/validation split (85%/15%)...")
strat = df["consonant"].astype(str) + "|" + df["vowel"].astype(str)
train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=0.15, random_state=SEED, stratify=strat
)

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)

# TF data loader
def decode_img(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=1)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="nearest")
    img = tf.cast(img, tf.float32) / 255.0
    return img


def make_ds(frame, shuffle=False):
    img_paths = frame["filename"].apply(lambda x: os.path.join(IMG_DIR, x)).tolist()
    cons = le_consonant.transform(frame["consonant"]).astype(np.int32)
    vow = le_vowel.transform(frame["vowel"]).astype(np.int32)

    ds_x = tf.data.Dataset.from_tensor_slices(img_paths).map(
        lambda p: decode_img(p), num_parallel_calls=tf.data.AUTOTUNE
    )
    ds_y1 = tf.data.Dataset.from_tensor_slices(cons)
    ds_y2 = tf.data.Dataset.from_tensor_slices(vow)
    ds = tf.data.Dataset.zip((ds_x, {"cons": ds_y1, "vow": ds_y2}))
    if shuffle:
        ds = ds.shuffle(4096, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(df_train, shuffle=True)
val_ds = make_ds(df_val, shuffle=False)

len_train = len(df_train)
len_val = len(df_val)
logger.info(f"Training samples: {len_train}")
logger.info(f"Validation samples: {len_val}")
print("Train samples:", len_train, " Val samples:", len_val)

### Data loading and preparation

This step:
- Reads `labels.csv` and shows a preview
- Fits two label encoders (`consonant`, `vowel`) and reports class counts
- Analyzes class balance for both heads and saves a figure
- Builds stratified train/validation splits using the combined label for better balance
- Creates efficient `tf.data` pipelines with caching, shuffling, batching, and prefetching

In [ ]:
# GPU tuning: enable mixed precision and memory growth
import tensorflow as tf
from tensorflow.keras import mixed_precision

# Use float16 on GPU for speed; losses/metrics handle casting automatically
mixed_precision.set_global_policy("mixed_float16")

# Avoid grabbing all GPU memory up front
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("Could not set memory growth on", gpu, e)

print("Policy:", mixed_precision.global_policy())
print("GPUs:", tf.config.list_logical_devices("GPU"))

### GPU setup: mixed precision and memory growth

If a compatible GPU is available, enabling mixed precision can significantly speed up training and reduce memory usage by using float16 on supported operations. We also enable memory growth so TensorFlow does not pre‑allocate all GPU memory at once.

- Mixed precision policy: `mixed_float16` (safe for this architecture; final Dense logits are kept in float32)
- Memory growth: allows dynamic allocation to play nicely with other processes

In [ ]:
# Build two-headed CNN
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
x = layers.BatchNormalization(dtype="float32")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization(dtype="float32")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization(dtype="float32")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization(dtype="float32")(x)
x = layers.MaxPooling2D()(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

# Shared embedding
emb = layers.Dense(256, activation="relu")(x)
emb = layers.Dropout(0.3)(emb)

# Head 1: consonant
cons_logits = layers.Dense(num_consonant_classes, name="cons", dtype="float32")(emb)
# Head 2: vowel sign
vow_logits = layers.Dense(num_vowel_classes, name="vow", dtype="float32")(emb)

model = keras.Model(inputs=inputs, outputs={"cons": cons_logits, "vow": vow_logits})

losses = {
    "cons": keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    "vow": keras.losses.SparseCategoricalCrossentropy(from_logits=True),
}
metrics = {
    "cons": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    "vow": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
}

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=losses,
    metrics=metrics,
)

model.summary()

### Model architecture: shared CNN with two heads

We use a compact CNN backbone with BatchNorm and pooling followed by a shared embedding. Two output heads (Dense layers) produce logits for:
- `cons` (consonant head): size = `num_consonant_classes`
- `vow` (vowel head): size = `num_vowel_classes`

Training objective
- Two SparseCategoricalCrossentropy losses (from logits) optimized jointly
- Metrics: accuracy reported per head

Notes
- We set Dense outputs to `dtype='float32'` to avoid mixed‑precision loss/metric instability
- GlobalAveragePooling keeps the model parameter count small and efficient

In [ ]:
# Training configuration with improved callbacks
EPOCHS = 25
logger.info(f"Starting training for {EPOCHS} epochs...")

# Enhanced callbacks with learning rate scheduling
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="malayalam_dataset/best_model_{epoch:02d}_{val_loss:.4f}.keras",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", 
        patience=7,  # Increased patience for more epochs
        restore_best_weights=True,
        verbose=1,
        min_delta=1e-4
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", 
        factor=0.5, 
        patience=4, 
        min_lr=1e-6,
        verbose=1,
        cooldown=2
    ),
    tf.keras.callbacks.CSVLogger(
        "malayalam_dataset/training_log.csv", 
        append=False
    ),
    tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: logger.info(
            f"Epoch {epoch+1}: loss={logs['loss']:.4f}, val_loss={logs['val_loss']:.4f}, "
            f"cons_acc={logs['cons_acc']:.4f}, vow_acc={logs['vow_acc']:.4f}"
        )
    )
]

logger.info("Starting model training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

logger.info("Training completed successfully!")
print("Training complete.")

### Training strategy and callbacks

We train for up to 25 epochs with the following callbacks:
- ModelCheckpoint: saves the best `.keras` model by lowest `val_loss` each epoch
- EarlyStopping: monitors `val_loss` with patience 7 and restores best weights
- ReduceLROnPlateau: halves LR on plateau (patience 4, cooldown 2, min LR 1e‑6)
- CSVLogger: writes per‑epoch metrics to `malayalam_dataset/training_log.csv`
- LambdaCallback: logs concise metrics after each epoch

This setup helps stabilize training, prevents overfitting, and produces versioned artifacts for later inference.

In [ ]:
# Comprehensive training visualization
def plot_training_history(history):
    """Plot comprehensive training history with all metrics"""
    logger.info("Creating training history visualizations...")
    
    fig, axes = plt.subplots(3, 2, figsize=(20, 18))
    
    # Training and validation loss
    axes[0,0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0,0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0,0].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].set_ylabel('Loss')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Consonant accuracy
    axes[0,1].plot(history.history['cons_acc'], label='Training Consonant Acc', linewidth=2)
    axes[0,1].plot(history.history['val_cons_acc'], label='Validation Consonant Acc', linewidth=2)
    axes[0,1].set_title('Consonant Accuracy', fontsize=14, fontweight='bold')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('Accuracy')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Vowel accuracy
    axes[1,0].plot(history.history['vow_acc'], label='Training Vowel Acc', linewidth=2)
    axes[1,0].plot(history.history['val_vow_acc'], label='Validation Vowel Acc', linewidth=2)
    axes[1,0].set_title('Vowel Accuracy', fontsize=14, fontweight='bold')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_ylabel('Accuracy')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # Consonant loss
    axes[1,1].plot(history.history['cons_loss'], label='Training Consonant Loss', linewidth=2)
    axes[1,1].plot(history.history['val_cons_loss'], label='Validation Consonant Loss', linewidth=2)
    axes[1,1].set_title('Consonant Loss', fontsize=14, fontweight='bold')
    axes[1,1].set_xlabel('Epoch')
    axes[1,1].set_ylabel('Loss')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # Vowel loss
    axes[2,0].plot(history.history['vow_loss'], label='Training Vowel Loss', linewidth=2)
    axes[2,0].plot(history.history['val_vow_loss'], label='Validation Vowel Loss', linewidth=2)
    axes[2,0].set_title('Vowel Loss', fontsize=14, fontweight='bold')
    axes[2,0].set_xlabel('Epoch')
    axes[2,0].set_ylabel('Loss')
    axes[2,0].legend()
    axes[2,0].grid(True, alpha=0.3)
    
    # Learning rate (if available)
    if 'lr' in history.history:
        axes[2,1].plot(history.history['lr'], label='Learning Rate', linewidth=2, color='red')
        axes[2,1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
        axes[2,1].set_xlabel('Epoch')
        axes[2,1].set_ylabel('Learning Rate')
        axes[2,1].set_yscale('log')
        axes[2,1].legend()
        axes[2,1].grid(True, alpha=0.3)
    else:
        # Combined accuracy comparison
        axes[2,1].plot(history.history['cons_acc'], label='Consonant Acc', linewidth=2)
        axes[2,1].plot(history.history['vow_acc'], label='Vowel Acc', linewidth=2)
        axes[2,1].plot(history.history['val_cons_acc'], label='Val Consonant Acc', linewidth=2, linestyle='--')
        axes[2,1].plot(history.history['val_vow_acc'], label='Val Vowel Acc', linewidth=2, linestyle='--')
        axes[2,1].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
        axes[2,1].set_xlabel('Epoch')
        axes[2,1].set_ylabel('Accuracy')
        axes[2,1].legend()
        axes[2,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('malayalam_dataset/training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print final metrics
    final_epoch = len(history.history['loss'])
    print(f"\n🎯 Final Training Results (Epoch {final_epoch}):")
    print(f"   Training Loss: {history.history['loss'][-1]:.4f}")
    print(f"   Validation Loss: {history.history['val_loss'][-1]:.4f}")
    print(f"   Training Consonant Acc: {history.history['cons_acc'][-1]:.4f}")
    print(f"   Validation Consonant Acc: {history.history['val_cons_acc'][-1]:.4f}")
    print(f"   Training Vowel Acc: {history.history['vow_acc'][-1]:.4f}")
    print(f"   Validation Vowel Acc: {history.history['val_vow_acc'][-1]:.4f}")
    
    # Find best epoch
    best_epoch = np.argmin(history.history['val_loss']) + 1
    best_val_loss = min(history.history['val_loss'])
    print(f"\n🏆 Best Validation Loss: {best_val_loss:.4f} at epoch {best_epoch}")

# Plot training history
plot_training_history(history)

### Interpreting the training history plots

The following figure summarizes model behavior across epochs:
- Loss: overall objective (should trend down) and per‑head losses
- Accuracy: per‑head training and validation accuracy curves
- LR/Comparison: shows learning‑rate schedule (if tracked) or a side‑by‑side accuracy comparison

What to look for
- Diverging train vs val curves → potential overfitting
- Plateaus → LR reduction triggers; consider longer training or architecture tweaks
- Stability → smoother curves often indicate good regularization and sufficient data

In [ ]:
# Comprehensive model evaluation and performance analysis
logger.info("Starting comprehensive model evaluation...")

# Evaluate model
val_metrics = model.evaluate(val_ds, verbose=1)
logger.info(f"Validation metrics: {val_metrics}")
print("Validation metrics:", val_metrics)

# Inference helper converts model outputs back to labels
inv_consonant = {i: c for i, c in enumerate(le_consonant.classes_)}
inv_vowel = {i: v for i, v in enumerate(le_vowel.classes_)}

# Get predictions for entire validation set
logger.info("Generating predictions for validation set...")
all_imgs = []
all_cons_true = []
all_vow_true = []
all_cons_pred = []
all_vow_pred = []

for batch in val_ds:
    imgs, ytrue = batch
    logits = model.predict(imgs, verbose=0)
    
    cons_pred = tf.argmax(logits["cons"], axis=-1).numpy()
    vow_pred = tf.argmax(logits["vow"], axis=-1).numpy()
    
    all_imgs.extend(imgs.numpy())
    all_cons_true.extend(ytrue["cons"].numpy())
    all_vow_true.extend(ytrue["vow"].numpy())
    all_cons_pred.extend(cons_pred)
    all_vow_pred.extend(vow_pred)

all_imgs = np.array(all_imgs)
all_cons_true = np.array(all_cons_true)
all_vow_true = np.array(all_vow_true)
all_cons_pred = np.array(all_cons_pred)
all_vow_pred = np.array(all_vow_pred)

logger.info(f"Processed {len(all_cons_true)} validation samples")

# Calculate accuracies
cons_accuracy = np.mean(all_cons_true == all_cons_pred)
vow_accuracy = np.mean(all_vow_true == all_vow_pred)
combined_accuracy = np.mean((all_cons_true == all_cons_pred) & (all_vow_true == all_vow_pred))

print(f"\n📊 Detailed Accuracy Metrics:")
print(f"   Consonant Accuracy: {cons_accuracy:.4f} ({cons_accuracy*100:.2f}%)")
print(f"   Vowel Accuracy: {vow_accuracy:.4f} ({vow_accuracy*100:.2f}%)")
print(f"   Combined Accuracy: {combined_accuracy:.4f} ({combined_accuracy*100:.2f}%)")

# Confusion matrices and detailed analysis
def create_confusion_matrices():
    logger.info("Creating confusion matrices...")
    
    fig, axes = plt.subplots(1, 2, figsize=(25, 10))
    
    # Consonant confusion matrix
    cons_cm = confusion_matrix(all_cons_true, all_cons_pred)
    
    # Limit to top classes for readability
    top_cons_classes = 30  # Show top 30 classes
    if len(le_consonant.classes_) > top_cons_classes:
        # Get most frequent classes
        unique_true, counts = np.unique(all_cons_true, return_counts=True)
        top_indices = unique_true[np.argsort(counts)[-top_cons_classes:]]
        
        # Filter confusion matrix
        mask = np.isin(all_cons_true, top_indices) & np.isin(all_cons_pred, top_indices)
        filtered_true = all_cons_true[mask]
        filtered_pred = all_cons_pred[mask]
        
        # Remap indices for filtered data
        idx_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted(top_indices))}
        filtered_true_remapped = np.array([idx_map[x] for x in filtered_true])
        filtered_pred_remapped = np.array([idx_map[x] for x in filtered_pred])
        
        cons_cm_filtered = confusion_matrix(filtered_true_remapped, filtered_pred_remapped)
        cons_labels = [le_consonant.classes_[i] for i in sorted(top_indices)]
    else:
        cons_cm_filtered = cons_cm
        cons_labels = le_consonant.classes_
    
    # Plot consonant confusion matrix
    sns.heatmap(cons_cm_filtered, annot=False, fmt='d', cmap='Blues', 
                xticklabels=cons_labels, yticklabels=cons_labels, ax=axes[0])
    axes[0].set_title(f'Consonant Confusion Matrix (Top {min(top_cons_classes, len(cons_labels))} Classes)', 
                      fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].tick_params(axis='both', labelsize=8)
    
    # Vowel confusion matrix
    vow_cm = confusion_matrix(all_vow_true, all_vow_pred)
    sns.heatmap(vow_cm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=le_vowel.classes_, yticklabels=le_vowel.classes_, ax=axes[1])
    axes[1].set_title('Vowel Confusion Matrix', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    axes[1].tick_params(axis='both', labelsize=10)
    
    plt.tight_layout()
    plt.savefig('malayalam_dataset/confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return cons_cm, vow_cm

cons_cm, vow_cm = create_confusion_matrices()

# Detailed classification reports
def print_classification_reports():
    logger.info("Generating detailed classification reports...")
    
    print("\n" + "="*80)
    print("📈 CONSONANT CLASSIFICATION REPORT")
    print("="*80)
    
    cons_report = classification_report(
        all_cons_true, all_cons_pred, 
        target_names=le_consonant.classes_, 
        zero_division=0,
        output_dict=True
    )
    
    # Print summary statistics
    print(f"Accuracy: {cons_report['accuracy']:.4f}")
    print(f"Macro Avg Precision: {cons_report['macro avg']['precision']:.4f}")
    print(f"Macro Avg Recall: {cons_report['macro avg']['recall']:.4f}")
    print(f"Macro Avg F1-Score: {cons_report['macro avg']['f1-score']:.4f}")
    print(f"Weighted Avg F1-Score: {cons_report['weighted avg']['f1-score']:.4f}")
    
    print("\n" + "="*80)
    print("📈 VOWEL CLASSIFICATION REPORT")
    print("="*80)
    
    vow_report = classification_report(
        all_vow_true, all_vow_pred,
        target_names=le_vowel.classes_,
        zero_division=0,
        output_dict=True
    )
    
    print(f"Accuracy: {vow_report['accuracy']:.4f}")
    print(f"Macro Avg Precision: {vow_report['macro avg']['precision']:.4f}")
    print(f"Macro Avg Recall: {vow_report['macro avg']['recall']:.4f}")
    print(f"Macro Avg F1-Score: {vow_report['macro avg']['f1-score']:.4f}")
    print(f"Weighted Avg F1-Score: {vow_report['weighted avg']['f1-score']:.4f}")
    
    return cons_report, vow_report

cons_report, vow_report = print_classification_reports()

# Sample predictions visualization
def visualize_sample_predictions(num_samples=16):
    logger.info(f"Visualizing {num_samples} sample predictions...")
    
    # Select random samples
    indices = np.random.choice(len(all_imgs), num_samples, replace=False)
    
    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    axes = axes.ravel()
    
    for i, idx in enumerate(indices):
        axes[i].imshow(all_imgs[idx].squeeze(), cmap='gray')
        
        # Get true and predicted labels
        true_cons = inv_consonant[all_cons_true[idx]]
        pred_cons = inv_consonant[all_cons_pred[idx]]
        true_vow = inv_vowel[all_vow_true[idx]]
        pred_vow = inv_vowel[all_vow_pred[idx]]
        
        # Determine if prediction is correct
        cons_correct = all_cons_true[idx] == all_cons_pred[idx]
        vow_correct = all_vow_true[idx] == all_vow_pred[idx]
        
        # Color code: green if correct, red if wrong
        cons_color = 'green' if cons_correct else 'red'
        vow_color = 'green' if vow_correct else 'red'
        
        title = f"True: {true_cons}+{true_vow}\nPred: {pred_cons}+{pred_vow}"
        axes[i].set_title(title, fontsize=10, 
                         color='black' if (cons_correct and vow_correct) else 'red')
        axes[i].axis('off')
    
    plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('malayalam_dataset/sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_sample_predictions()

# Print sample predictions (first 8)
print(f"\n🔍 Sample predictions (first 8):")
for i in range(min(8, len(all_cons_pred))):
    cons_match = "✓" if all_cons_true[i] == all_cons_pred[i] else "✗"
    vow_match = "✓" if all_vow_true[i] == all_vow_pred[i] else "✗"
    
    print(f"{i+1:2d}. Pred: {inv_consonant[all_cons_pred[i]]}+{inv_vowel[all_vow_pred[i]]} {cons_match}{vow_match} "
          f"| True: {inv_consonant[all_cons_true[i]]}+{inv_vowel[all_vow_true[i]]}")

logger.info("Model evaluation completed successfully!")

### Evaluation: metrics, confusion matrices, and sample predictions

We evaluate on the validation set and report:
- Head accuracies: consonant, vowel, and combined correctness (both heads correct)
- Confusion matrices: per‑head confusion to diagnose common confusions
- Classification reports: precision/recall/F1 per class and aggregate
- Sample predictions grid: quick visual audit of successes and failures

Artifacts are saved under `malayalam_dataset/` for review and sharing.